# Chapter 3 · Sections 3.7–3.8
# Uniform Persistence & Metzler Matrices and Monotone Systems

**Source:** Li, M.Y. (2018), pp. 95–101.

The final two sections of Chapter 3. Section 3.7 formalizes "the disease stays endemic" as a
precise mathematical property (uniform persistence) and proves it directly for Sec. 2.3's model.
Section 3.8 formalizes the monotone-systems machinery previewed in Sec. 2.5's Ross–MacDonald
model, culminating in Theorem 3.8.5 — the exact general theorem that section anticipated.


---
## Block 1 (§3.7) — Uniform persistence, proven directly for Sec. 2.3's model

### 1. Rewrite

**Uniform persistence** formalizes "the disease stays endemic, robustly" — not just
$\liminf I(t)>0$ for one particular solution, but a *single* positive bound $\epsilon_0$ that
works for *every* solution starting in the feasible region $D$: every solution's distance to the
boundary $\partial D$ stays above $\epsilon_0$ forever (3.18). **Theorem 3.7.1** gives sufficient
conditions in terms of the boundary's dynamics: if the largest compact invariant set $M$ on
$\partial D$ is *isolated* (has a neighborhood containing no other full orbit) and its stable set
$W^s(M)$ stays entirely within the boundary (nothing from the interior gets pulled onto $M$), then
uniform persistence follows. Applying this directly to Sec. 2.3's model (2.25): the feasible region
$\Gamma$'s boundary's only compact invariant set is $\{P_0\}$ (the disease-free equilibrium), and
$P_0$ is isolated; the key remaining question is whether $W^s(P_0)$ stays confined to the boundary
— equivalently, whether $P_0$ *repels* into the interior. Exactly the Lyapunov calculation from
Sec. 2.3, Block 4 (reused here almost verbatim!) shows: for initial conditions near $P_0$ with
$S_0$ close to 1, $\dot L=\beta I(S-1/\mathcal{R}_0)>0$ whenever $S_0>1/\mathcal{R}_0$ — true
whenever $\mathcal{R}_0>1$ — so $I(t)$ initially *increases* near $P_0$, confirming the repelling
property. This proves **Theorem 3.7.2**: the SIR model (2.25) is uniformly persistent if and only
if $\mathcal{R}_0>1$ — a clean, complete characterization.

### 2. Explain Like I'm 10

"The disease is endemic" sounds like a yes/no fact, but there's a subtlety: maybe the disease
*technically* never fully dies out, but gets closer and closer to zero as time passes (like an
infinitely-approaching-but-never-arriving situation) — that would feel more like "dying out slowly"
than genuinely "staying endemic." Uniform persistence rules out that loophole: it guarantees the
sick population never drops below some *fixed, positive* floor, no matter how long you wait, and
no matter which starting point you began from. It's the mathematical version of "this disease
isn't just barely hanging on — it has a genuine, permanent foothold."

### 3. Key Ideas

- **Uniform persistence is a UNIFORM statement across all initial conditions** — a stronger,
  more robust notion than persistence for any single trajectory.
- **Boundary dynamics determine interior persistence** — a beautiful, general principle: whether
  the interior dynamics "stick around" is governed by whether the *boundary's own* invariant sets
  repel.
- **This section's proof directly reuses Sec. 2.3's Lyapunov calculation** — a strong illustration
  that the same piece of mathematical work can serve two different theoretical purposes (there:
  global stability of $P_0$ when $\mathcal{R}_0\le1$; here: uniform persistence when
  $\mathcal{R}_0>1$ — genuinely complementary regimes of the SAME calculation).
- **Theorem 3.7.2 is an "if and only if"** — a complete characterization, not just a sufficient
  condition.

### 7. Visual Understanding — confirming Theorem 3.7.2 by direct simulation


In [1]:
# Directly testing Theorem 3.7.2: for R0>1, EVERY tested trajectory (starting
# from many different points near the boundary) should have liminf S(t),
# liminf I(t), and liminf (1-S(t)-I(t)) all staying above a common positive
# floor -- confirming uniform (not just eventual) persistence.
import numpy as np
from scipy.integrate import solve_ivp

def rhs(t, y, b, beta, gamma):
    S, I = y
    return [b - beta*I*S - b*S, beta*I*S - gamma*I - b*I]

b, gamma = 0.02, 0.2
beta_above = 0.5   # R0 = 0.5/0.22 > 1
R0 = beta_above / (b + gamma)
print(f"R0 = {R0:.3f} (> 1, testing uniform persistence)")

# Test many initial conditions, INCLUDING some very close to the boundary
starts = [(0.999, 0.0005), (0.001, 0.001), (0.5, 0.001), (0.001, 0.5), (0.5, 0.499), (0.9,0.09)]
t_tail = np.linspace(400, 600, 500)   # look at the LONG-RUN tail, not transients

min_S_liminf, min_I_liminf, min_gap_liminf = np.inf, np.inf, np.inf
for S0, I0 in starts:
    sol = solve_ivp(rhs, (0, 600), [S0, I0], args=(b, beta_above, gamma),
                     t_eval=t_tail, rtol=1e-11, atol=1e-11)
    S_tail, I_tail = sol.y
    gap_tail = 1 - S_tail - I_tail
    min_S_liminf = min(min_S_liminf, S_tail.min())
    min_I_liminf = min(min_I_liminf, I_tail.min())
    min_gap_liminf = min(min_gap_liminf, gap_tail.min())
    print(f"  start=({S0},{I0}): tail S in [{S_tail.min():.4f},{S_tail.max():.4f}], "
          f"tail I in [{I_tail.min():.4f},{I_tail.max():.4f}]")

print(f"\nAcross ALL {len(starts)} trajectories, minimum tail values:")
print(f"  min S(t) (long-run): {min_S_liminf:.4f}")
print(f"  min I(t) (long-run): {min_I_liminf:.4f}")
print(f"  min (1-S-I)(t) (long-run): {min_gap_liminf:.4f}")
print(f"All strictly positive and bounded away from 0 -> consistent with UNIFORM persistence")


R0 = 2.273 (> 1, testing uniform persistence)
  start=(0.999,0.0005): tail S in [0.4399,0.4400], tail I in [0.0509,0.0509]
  start=(0.001,0.001): tail S in [0.4399,0.4401], tail I in [0.0509,0.0509]
  start=(0.5,0.001): tail S in [0.4400,0.4400], tail I in [0.0509,0.0509]
  start=(0.001,0.5): tail S in [0.4400,0.4400], tail I in [0.0509,0.0509]
  start=(0.5,0.499): tail S in [0.4400,0.4400], tail I in [0.0509,0.0509]
  start=(0.9,0.09): tail S in [0.4400,0.4400], tail I in [0.0509,0.0509]

Across ALL 6 trajectories, minimum tail values:
  min S(t) (long-run): 0.4399
  min I(t) (long-run): 0.0509
  min (1-S-I)(t) (long-run): 0.5090
All strictly positive and bounded away from 0 -> consistent with UNIFORM persistence


### 10. Verification, discussed

Every tested trajectory — including ones starting extremely close to the boundary (e.g.
$I_0=0.0005$, barely inside the feasible region) — settles into a long-run range where $S(t),
I(t),$ and $1-S(t)-I(t)$ all stay comfortably bounded away from zero. Since all six very
different starting points converge to essentially the *same* tail range (consistent with the
endemic equilibrium $P^*$ being globally stable, per Sec. 2.3), a single shared $\epsilon_0$
(smaller than the smallest of the observed minimums) works for all of them simultaneously —
exactly the uniform persistence property Theorem 3.7.2 predicts for $\mathcal{R}_0>1$.


In [2]:
# Contrast case: R0 < 1 -- persistence should FAIL (I(t) -> 0, not bounded away from 0)
beta_below = 0.15
R0_below = beta_below / (b + gamma)
print(f"R0 = {R0_below:.3f} (< 1, persistence should FAIL)")

sol = solve_ivp(rhs, (0, 600), [0.5, 0.3], args=(b, beta_below, gamma),
                 t_eval=t_tail, rtol=1e-11, atol=1e-11)
I_tail_below = sol.y[1]
print(f"Tail I(t) range when R0<1: [{I_tail_below.min():.6f}, {I_tail_below.max():.6f}]")
print("I(t) -> 0, NOT bounded away from 0 -> persistence correctly FAILS, matching Theorem 3.7.2's 'only if' direction")


R0 = 0.682 (< 1, persistence should FAIL)
Tail I(t) range when R0<1: [0.000000, 0.000000]
I(t) -> 0, NOT bounded away from 0 -> persistence correctly FAILS, matching Theorem 3.7.2's 'only if' direction


### 10. Verification, discussed (continued)

When $\mathcal{R}_0<1$, $I(t)$'s tail values shrink toward essentially zero (to the precision
shown) — persistence genuinely fails, exactly matching the "only if" half of Theorem 3.7.2's
if-and-only-if characterization: uniform persistence holds precisely when, and only when,
$\mathcal{R}_0>1$.


---
## Block 2 (§3.8) — Metzler matrices, Perron–Frobenius, and monotone systems

### 1. Rewrite

A matrix is **Metzler** if every off-diagonal entry is non-negative (already used, without the
name, in Sec. 2.5). The **spectral radius** $\rho(A)$ is the largest eigenvalue modulus; the
**stability modulus** $s(A)$ is the largest eigenvalue *real part* — for a general matrix these are
different numbers, but for Metzler matrices specifically, the **Perron–Frobenius Theorem for
Metzler Matrices** (3.8.2) guarantees $s(A)$ itself is always an eigenvalue (not just an upper
bound on real parts), with a non-negative eigenvector — and if $A$ is additionally *irreducible*
(its associated directed graph is strongly connected — no subset of coordinates is "cut off" from
the rest), $s(A)$ is a *simple* eigenvalue with a strictly *positive* eigenvector. **Theorem 3.8.3**
collects several equivalent characterizations of "$A$ is stable" ($s(A)<0$) for Metzler matrices,
purely in terms of sign conditions — no eigenvalue computation needed. **Theorem 3.8.4** formalizes
exactly what Sec. 2.5 checked numerically: a system is monotone if and only if its Jacobian is
Metzler *everywhere*. Finally, **Theorem 3.8.5** is the general global-stability theorem Sec. 2.5
anticipated: for a strongly monotone, strictly sublinear system with bounded, non-negative-orthant-
invariant solutions, either everything converges to $0$, or there's a unique positive equilibrium
that is globally asymptotically stable — *exactly* the dichotomy observed numerically in Sec. 2.5's
$\mathcal{R}_0<1$ vs. $\mathcal{R}_0>1$ cases.

### 2. Explain Like I'm 10

Remember the "sick classrooms can't leapfrog past healthier ones" idea from Sec. 2.5? This section
gives it a name (monotone systems) and connects it to a special *kind* of matrix (Metzler) whose
defining feature — no negative "cross-talk" between different coordinates — is exactly what
prevents that leapfrogging. The Perron–Frobenius theorem is a powerful guarantee: for these
particular well-behaved (Metzler) matrices, the "most important" eigenvalue (the one controlling
long-run growth or decay) isn't just some abstract complex number that might be hard to interpret —
it's guaranteed to be *achievable by an actual non-negative direction*, which is a much more
concrete, usable guarantee than what you'd get for a general, arbitrary matrix.

### 4. Mathematical Breakdown — verifying Sec. 2.5's Jacobian is genuinely Metzler, using Theorem 3.8.4

Recall Sec. 2.5's Jacobian: $J(x,y)=\begin{pmatrix}-\gamma_1-amb_1y&amb_1(1-x)\\ab_2(1-y)&
-\gamma_2-ab_2x\end{pmatrix}$. Theorem 3.8.4 says the Ross–MacDonald system is monotone
**if and only if** this matrix is Metzler *throughout* the feasible region $\Gamma=[0,1]^2$ — i.e.
both off-diagonal entries $amb_1(1-x)$ and $ab_2(1-y)$ must be $\ge0$ everywhere in $\Gamma$. Since
$a,m,b_1,b_2>0$ (positive rate/probability parameters) and $1-x,1-y\ge0$ for $(x,y)\in[0,1]^2$,
both off-diagonal entries are indeed non-negative everywhere — confirming (via the *general*
theorem, not just the numerical spot-check from Sec. 2.5) that the Ross–MacDonald system genuinely
is monotone throughout its feasible region.

### 5. Symbols

| Symbol | Meaning | Notes |
|---|---|---|
| $\rho(A)$ | Spectral radius: $\max|\lambda_i|$ | Largest eigenvalue MODULUS |
| $s(A)$ | Stability modulus: $\max\text{Re}(\lambda_i)$ | Largest eigenvalue REAL PART — the more relevant quantity for stability |
| Irreducible | The matrix's directed graph is strongly connected | Ensures the Perron eigenvalue is simple with a strictly positive eigenvector |
| Strongly monotone | $x\le y,\ x\ne y \Rightarrow \phi_t(x)<\phi_t(y)$ | The flow strictly preserves (strict) ordering |
| Sublinear / strictly sublinear | $f(\lambda x)\ge\lambda f(x)$ / $>$ for $0<\lambda<1$ | Exactly the property checked numerically in Sec. 2.5 |


In [3]:
# Verifying the Perron-Frobenius Theorem for Metzler matrices numerically:
# generate random Metzler matrices, confirm s(A) is genuinely an eigenvalue
# (not just an upper bound), and that its eigenvector is non-negative.
import numpy as np

rng = np.random.default_rng(1)

def random_metzler(n, rng, diag_range=(-5,-0.1), offdiag_range=(0,3)):
    A = rng.uniform(*offdiag_range, size=(n,n))
    np.fill_diagonal(A, rng.uniform(*diag_range, size=n))
    return A

violations = 0
n_tests = 500
for _ in range(n_tests):
    n = rng.integers(2, 6)
    A = random_metzler(n, rng)
    eigvals, eigvecs = np.linalg.eig(A)
    s_A = eigvals[np.argmax(eigvals.real)].real

    # Check s(A) itself is (very nearly) an eigenvalue with near-zero imaginary part
    matching = np.argmin(np.abs(eigvals - s_A))
    is_eigenvalue = np.abs(eigvals[matching].imag) < 1e-8

    # Check its eigenvector can be taken non-negative (up to an overall sign flip)
    v = eigvecs[:, matching].real
    nonneg = np.all(v >= -1e-8) or np.all(v <= 1e-8)

    if not (is_eigenvalue and nonneg):
        violations += 1

print(f"Perron-Frobenius (Metzler) check: {violations} violations out of {n_tests} random Metzler matrices")
print("(should be 0: s(A) is always achieved by a real eigenvalue with a sign-definite eigenvector)")


Perron-Frobenius (Metzler) check: 0 violations out of 500 random Metzler matrices
(should be 0: s(A) is always achieved by a real eigenvalue with a sign-definite eigenvector)


### 10. Verification

Across 500 randomly generated Metzler matrices of varying sizes, the Perron–Frobenius property
holds with **zero violations**: the stability modulus $s(A)$ is always achieved by a genuine
(real, not complex) eigenvalue, and its eigenvector can always be taken entirely non-negative —
exactly as Theorem 3.8.2 guarantees, confirmed here far beyond the single $2\times2$ example from
Sec. 2.5.


In [4]:
# Verifying Theorem 3.8.3's equivalence (1) <=> (2): for Metzler matrices,
# "A is stable" (s(A)<0) should be exactly equivalent to "-A^{-1} >= 0".
import numpy as np

rng2 = np.random.default_rng(7)
agreements = 0
n_tests = 500
for _ in range(n_tests):
    n = rng2.integers(2, 5)
    A = random_metzler(n, rng2, diag_range=(-8, 2), offdiag_range=(0, 3))  # allow both stable & unstable cases
    s_A = np.max(np.linalg.eigvals(A).real)
    is_stable = s_A < 0
    try:
        minus_Ainv = -np.linalg.inv(A)
        is_nonneg_inv = np.all(minus_Ainv >= -1e-8)
    except np.linalg.LinAlgError:
        is_nonneg_inv = False  # singular -> can't be stable & invertible
    if is_stable == is_nonneg_inv:
        agreements += 1

print(f"Theorem 3.8.3 equivalence (1)<=>(2): {agreements}/{n_tests} random Metzler matrices agree")


Theorem 3.8.3 equivalence (1)<=>(2): 500/500 random Metzler matrices agree


### 10. Verification, discussed

Across 500 more random Metzler matrices (this time deliberately spanning both stable and unstable
cases), "$A$ is stable" and "$-A^{-1}\ge0$" agree in every single trial — directly confirming
Theorem 3.8.3's equivalence (1)⟺(2) numerically, not just algebraically.

### 7. Visual Understanding — Theorem 3.8.5 applied back to Sec. 2.5


In [5]:
# Directly connecting Theorem 3.8.5's dichotomy to Sec. 2.5's own numerically
# observed R0<1 / R0>1 split -- confirming BOTH cases of the theorem's
# conclusion (1): "either all solutions -> 0, or there's a globally stable p>0".
import numpy as np
from scipy.integrate import solve_ivp

def ross_macdonald(t, state, a, m, b1, b2, gamma1, gamma2):
    x, y = state
    dx = a*m*b1*y*(1-x) - gamma1*x
    dy = a*b2*x*(1-y) - gamma2*y
    return [dx, dy]

a_param, b1, b2, gamma1, gamma2 = 0.3, 0.5, 0.3, 0.1, 0.05

for m_param, label in [(0.3, "R0 < 1 (case: all solutions -> 0)"), (6.0, "R0 > 1 (case: globally stable p>0)")]:
    R0 = a_param**2 * m_param * b1 * b2 / (gamma1 * gamma2)
    t_end = 3000 if R0 < 1 else 500  # R0<1 decays only algebraically slowly near 0, needs more time
    long_run_vals = []
    for x0, y0 in [(0.1,0.1),(0.9,0.05),(0.05,0.9),(0.5,0.5)]:
        sol = solve_ivp(ross_macdonald, (0, t_end), [x0, y0], args=(a_param, m_param, b1, b2, gamma1, gamma2),
                         t_eval=[t_end], rtol=1e-11, atol=1e-11)
        long_run_vals.append(sol.y[:, -1])
    long_run_vals = np.array(long_run_vals)
    print(f"{label}: R0={R0:.2f}")
    print(f"  Long-run (x,y) from 4 different starting points:\n{long_run_vals}")
    print(f"  All essentially identical? {np.allclose(long_run_vals, long_run_vals[0], atol=1e-4)}\n")


R0 < 1 (case: all solutions -> 0): R0=0.81
  Long-run (x,y) from 4 different starting points:
[[7.77278064e-11 1.66060431e-10]
 [1.34156439e-10 2.83357563e-10]
 [1.38199024e-10 2.95589357e-10]
 [1.34997431e-10 2.86102820e-10]]
  All essentially identical? True



R0 > 1 (case: globally stable p>0): R0=16.20
  Long-run (x,y) from 4 different starting points:
[[0.84444444 0.6031746 ]
 [0.84444444 0.6031746 ]
 [0.84444444 0.6031746 ]
 [0.84444444 0.6031746 ]]
  All essentially identical? True



### 10. Verification, discussed

Both branches of Theorem 3.8.5's conclusion are confirmed directly: when $\mathcal{R}_0<1$, all
four tested trajectories converge to essentially $(0,0)$ regardless of starting point; when
$\mathcal{R}_0>1$, all four converge to the *same* nonzero point $(x^*,y^*)$ — confirming both the
existence of a globally attracting positive equilibrium AND that it is genuinely unique (every
different starting point lands on the identical point, to high numerical precision). This is
precisely the general theorem's conclusion, now seen as the *general, structural reason* behind
what Sec. 2.5 could previously only observe numerically for one specific example.


---
## Summary

**One sentence:** Section 3.7 formalizes "the disease stays endemic, robustly" as uniform
persistence and proves it holds for Sec. 2.3's model exactly when $\mathcal{R}_0>1$ (reusing that
section's own Lyapunov calculation for the proof), while Section 3.8 develops the general Metzler-
matrix/monotone-systems theory — including the Perron–Frobenius theorem for Metzler matrices and
Theorem 3.8.5's global-stability dichotomy — that Sec. 2.5's Ross–MacDonald model anticipated,
verified here to hold across hundreds of random matrices and confirmed directly against Sec. 2.5's
own two example regimes.

**One paragraph:** Uniform persistence strengthens "the infected population doesn't vanish" into a
uniform guarantee — a single positive floor $\epsilon_0$ that bounds every trajectory's distance
from the boundary, for all time. Theorem 3.7.1 reduces establishing this to checking the boundary's
own dynamics (an isolated invariant set there whose stable set stays confined to the boundary), and
applying this directly to Sec. 2.3's SIR-with-demography model — reusing that section's Lyapunov
derivative calculation almost verbatim — proves Theorem 3.7.2: uniform persistence holds if and
only if $\mathcal{R}_0>1$, confirmed here numerically by observing that every tested trajectory's
long-run tail stays uniformly bounded away from the boundary in the $\mathcal{R}_0>1$ case, and
fails to in the $\mathcal{R}_0<1$ case. Section 3.8 then develops Metzler matrices (already used
implicitly in Sec. 2.5) rigorously: the Perron–Frobenius Theorem for Metzler matrices (confirmed
here across 500 random matrices) guarantees the stability modulus is always a genuine eigenvalue
with a non-negative eigenvector; Theorem 3.8.3's equivalent stability characterizations were
verified numerically as well; and Theorem 3.8.4 confirms monotonicity is exactly equivalent to a
Metzler Jacobian everywhere — directly justifying Sec. 2.5's numerical Metzler check. Finally,
Theorem 3.8.5 formalizes the exact global-stability dichotomy (converge to zero, or to a unique
globally stable positive equilibrium) that Sec. 2.5 observed only numerically, confirmed here by
re-simulating both of that section's example regimes and showing every tested trajectory converges
to the identical long-run point regardless of starting condition.

**Bullet points:**
- Uniform persistence: a UNIFORM (single shared $\epsilon_0$) lower bound across all trajectories, not just eventual positivity for each one individually.
- Theorem 3.7.2 (SIR model persists iff $\mathcal{R}_0>1$) reuses Sec. 2.3's own Lyapunov calculation — confirmed numerically for both regimes.
- Metzler matrices: non-negative off-diagonal entries; Perron-Frobenius guarantees the stability modulus is a genuine eigenvalue with a non-negative eigenvector — verified across 500 random matrices.
- Theorem 3.8.3's equivalent stability characterizations verified numerically (500 more random matrices).
- Theorem 3.8.4: monotone $\iff$ Metzler Jacobian everywhere — directly justifies Sec. 2.5's numerical check.
- Theorem 3.8.5's global-stability dichotomy directly reproduced by re-simulating Sec. 2.5's own two example regimes.

## Memory Aids

- **Mnemonic**: "Persistence = a FLOOR, not just staying positive" — the uniform, shared lower
  bound is the whole point.
- **Mental model**: Metzler matrices are matrices with "no backstabbing" between coordinates — one
  variable can only ever help (never hurt) another's growth, off the diagonal.
- **Common mistake**: confusing spectral radius $\rho(A)$ (largest eigenvalue MODULUS) with
  stability modulus $s(A)$ (largest eigenvalue REAL PART) — they coincide only in special cases;
  $s(A)$ is the one that actually determines stability.
- **Common mistake**: forgetting that Theorem 3.8.5's dichotomy requires BOTH strong monotonicity
  AND strict sublinearity together — neither alone is sufficient for the global conclusion.
- **Exam tip**: know Theorem 3.8.4 cold — "monotone $\iff$ Metzler Jacobian everywhere" is the
  single fact connecting Sec. 2.5's numerical check to this section's general theory.

## Connections

- **← Section 2.3**: Section 3.7's proof of Theorem 3.7.2 is a direct, near-verbatim reuse of that
  section's own Lyapunov derivative calculation, now serving a complementary theoretical purpose.
- **← Section 2.5**: Section 3.8 is the formal general theory behind that section's numerical
  Metzler-matrix and sublinearity checks — Theorem 3.8.5 is exactly the theorem anticipated there.
- **→ Chapter 4**: uniform persistence is often a PREREQUISITE for meaningful parameter estimation
  from real disease data — you can only reliably fit an "endemic level" parameter if the model
  genuinely predicts a robust, non-vanishing endemic state.
- **→ Chapter 5**: monotone-systems theory extends naturally to the higher-dimensional models
  developed there, where full Lyapunov-function constructions become increasingly difficult by
  hand.
- **→ Economics**: Metzler matrices originate in economics (input-output/general equilibrium
  models) — the same mathematics describing "no backstabbing" between economic sectors applies
  unchanged to epidemic compartments.

*Practice problems for this section are in `../../chapter_03/exercises/section_3_7_3_8_exercises.md`, with full
worked solutions in `../../chapter_03/solutions/section_3_7_3_8_solutions.md`.*

---

# Chapter 3 Complete

All eight sections of Chapter 3 — the book's core mathematical toolkit — are now built, verified
numerically, and cross-checked against their concrete applications throughout Chapter 2.
